In [1]:
import re
import numpy as np
import pandas as pd


# ============================================================
# BLACK-BOX LLM ENVIRONMENTAL IMPACT ESTIMATOR
# ============================================================
#
# PIPELINE
#
# Black-box model tag
#       ↓
# HHAI-inspired model size classification
#       ↓
# Assumed GPU count
#       ↓
# Find comparable models from Joule TPS database
#       ↓
# Representative aggregate production TPS
#       ↓
# Joule-style Monte Carlo energy estimate
#       ↓
# Carbon = Energy × Carbon Intensity
#
#
# IMPORTANT:
# This is a COLLEGE-PROJECT ESTIMATOR.
#
# It produces scenario-based estimates.
# It does NOT measure the actual infrastructure used by
# OpenAI, Anthropic, Google, etc.
# ============================================================


N_SAMPLES = 10_000
RANDOM_SEED = 42


# Official Joule paper repository dataset.
JOULE_CSV_URL = (
    "https://raw.githubusercontent.com/"
    "microsoft/aienergy_simulation/main/"
    "model_throughput_DB.csv"
)

joule_df = pd.read_csv(JOULE_CSV_URL)

print("Joule TPS records:", len(joule_df))
print(joule_df.columns.tolist())

Joule TPS records: 25
['Model', 'Total Parameter Count (B)', 'Active Parameter Count (B)', 'Type (Dense / MoE)', 'Hardware', 'Scaled Power', 'TP Size', 'Quantization', 'Tokens per Second (TPS)', 'Input Length', 'Output Length', 'Batch Size', 'Power for Node (kW)', 'Source', 'Comments', 'Confidence level']


In [2]:
# ============================================================
# HHAI-INSPIRED MODEL CLASSIFICATION
# ============================================================
#
# HHAI parameter-size bands:
#
# Nano   : < 7B
# Micro  : 7B - 20B
# Small  : 20B - 40B
# Medium : 40B - 70B
# Large  : > 70B
#
#
# HHAI also makes heuristic assignments when parameters
# are not disclosed:
#
# Flagship proprietary model -> Large
# "mini" model              -> Medium
# "nano" model              -> Small
#
#
# ASSUMPTION MADE BY OUR PROJECT:
#
# We generalize this naming heuristic to arbitrary black-box
# model tags.
#
# Therefore classification from the model name is only a
# heuristic and NOT evidence of the model's true parameter
# count.
# ============================================================


CLASS_GPU_COUNT = {
    "Nano": 1,
    "Micro": 1,
    "Small": 2,
    "Medium": 4,
    "Large": 8
}


def classify_parameter_count(parameters_billion):
    """
    HHAI size buckets.
    """

    if parameters_billion < 7:
        return "Nano"

    elif parameters_billion < 20:
        return "Micro"

    elif parameters_billion < 40:
        return "Small"

    elif parameters_billion <= 70:
        return "Medium"

    else:
        return "Large"

In [3]:
def extract_parameter_count(model_tag):
    """
    Try to obtain a parameter count directly from a model name.

    Examples:
        llama-3.1-8b   -> 8
        qwen-32b       -> 32
        model-405b     -> 405

    Returns None when the model tag does not expose a
    parameter count.
    """

    model_tag = model_tag.lower()

    matches = re.findall(
        r"(\d+(?:\.\d+)?)\s*[bB]\b",
        model_tag
    )

    if not matches:
        return None

    # Usually the final number followed by B is the
    # parameter count.
    return float(matches[-1])

In [4]:
def classify_blackbox_model(model_tag):
    """
    Determine an HHAI-inspired hardware class from a model tag.

    Priority:

    1. If parameter count appears in the tag:
           use the HHAI parameter bands.

    2. If parameters are hidden:
           "nano"  -> Small
           "mini"  -> Medium
           other proprietary/flagship -> Large

    IMPORTANT:
    The Mini/Nano/flagship mappings are heuristic assumptions.
    """

    tag = model_tag.lower()

    # --------------------------------------------------------
    # STEP 1: Look for explicit parameter count
    # --------------------------------------------------------

    parameters_b = extract_parameter_count(tag)

    if parameters_b is not None:

        model_class = classify_parameter_count(
            parameters_b
        )

        classification_method = (
            f"Parameter count found in model tag "
            f"({parameters_b}B)"
        )

    # --------------------------------------------------------
    # STEP 2: HHAI-style naming heuristic
    # --------------------------------------------------------

    elif "nano" in tag:

        # HHAI classified GPT-4.1 nano as Small rather than
        # assuming the literal word "nano" means <7B.
        model_class = "Small"

        classification_method = (
            "Undisclosed parameters; "
            "HHAI-style Nano naming heuristic"
        )

    elif "mini" in tag:

        model_class = "Medium"

        classification_method = (
            "Undisclosed parameters; "
            "HHAI-style Mini naming heuristic"
        )

    # Optional very-small labels for generic open models.
    elif (
        "tiny" in tag
        or "micro" in tag
    ):

        model_class = "Micro"

        classification_method = (
            "Undisclosed parameters; "
            "project naming heuristic"
        )

    else:

        # HHAI puts undisclosed flagship OpenAI / Anthropic
        # models into the Large class.
        #
        # OUR EXTENSION:
        # Unknown black-box models are conservatively treated
        # as flagship/Large unless another clue is available.

        model_class = "Large"

        classification_method = (
            "Undisclosed parameters; "
            "assumed flagship/Large"
        )


    gpu_count = CLASS_GPU_COUNT[model_class]

    return {
        "model_tag": model_tag,
        "parameters_b": parameters_b,
        "class": model_class,
        "gpu_count": gpu_count,
        "classification_method": classification_method
    }

In [5]:
for model in [
    "llama-3.1-8b",
    "qwen2.5-32b",
    "llama-3.1-70b",
    "gpt-4o-mini",
    "gpt-4.1-nano",
    "claude-sonnet",
    "gpt-5.5"
]:

    print(classify_blackbox_model(model))

{'model_tag': 'llama-3.1-8b', 'parameters_b': 8.0, 'class': 'Micro', 'gpu_count': 1, 'classification_method': 'Parameter count found in model tag (8.0B)'}
{'model_tag': 'qwen2.5-32b', 'parameters_b': 32.0, 'class': 'Small', 'gpu_count': 2, 'classification_method': 'Parameter count found in model tag (32.0B)'}
{'model_tag': 'llama-3.1-70b', 'parameters_b': 70.0, 'class': 'Medium', 'gpu_count': 4, 'classification_method': 'Parameter count found in model tag (70.0B)'}
{'model_tag': 'gpt-4o-mini', 'parameters_b': None, 'class': 'Medium', 'gpu_count': 4, 'classification_method': 'Undisclosed parameters; HHAI-style Mini naming heuristic'}
{'model_tag': 'gpt-4.1-nano', 'parameters_b': None, 'class': 'Small', 'gpu_count': 2, 'classification_method': 'Undisclosed parameters; HHAI-style Nano naming heuristic'}
{'model_tag': 'claude-sonnet', 'parameters_b': None, 'class': 'Large', 'gpu_count': 8, 'classification_method': 'Undisclosed parameters; assumed flagship/Large'}
{'model_tag': 'gpt-5.5', '

In [10]:
# ============================================================
# PREPARE JOULE TPS DATABASE
# ============================================================
#
# The current Joule CSV uses descriptive column names.
# We rename the columns we need to simpler internal names.
#
# This does NOT modify the original CSV file.
# It only changes the names inside our pandas DataFrame.
# ============================================================


joule_df = joule_df.rename(columns={

    "Total Parameter Count (B)": "Total Parameter Count",

    "Active Parameter Count (B)": "Active Parameter Count",

    "Type (Dense / MoE)": "Model Type",

    "Tokens per Second (TPS)": "TPS",

    "Power for Node (kW)": "Node Power kW",

    "Confidence level": "Confidence"
})


# Convert required numerical columns to numbers.
numeric_columns = [
    "Total Parameter Count",
    "Active Parameter Count",
    "TPS",
    "Input Length",
    "Output Length",
    "Batch Size",
    "Node Power kW"
]


for column in numeric_columns:

    joule_df[column] = pd.to_numeric(
        joule_df[column],
        errors="coerce"
    )


# ============================================================
# CLASSIFY EACH JOULE MODEL USING HHAI SIZE CLASSES
# ============================================================


def joule_classify(params):

    if pd.isna(params):
        return None

    return classify_parameter_count(
        float(params)
    )


joule_df["HHAI_Class"] = (
    joule_df["Total Parameter Count"]
    .apply(joule_classify)
)


# Check the result
display(
    joule_df[
        [
            "Model",
            "Total Parameter Count",
            "Active Parameter Count",
            "Model Type",
            "Hardware",
            "TPS",
            "Input Length",
            "Output Length",
            "HHAI_Class"
        ]
    ]
)

,Model,Total Parameter Count,Active Parameter Count,Model Type,Hardware,TPS,Input Length,Output Length,HHAI_Class
0,Mixtral 8x22B,141,22.0,MoE,H100 80GB HBM3,1177.80,NaN,385.22,Large
1,Mistral Large,123,NaN,Dense,H100 80GB HBM3,1159.78,NaN,510.32,Large
2,Llama 3.1 405B,405,NaN,Dense,H100 80GB HBM3,579.63,NaN,449.80,Large
3,Llama 3.1 405B,405,NaN,Dense,H100 80GB HBM3,3732.40,128.0,128.00,Large
4,Llama 3.1 405B,405,NaN,Dense,H100 80GB HBM3,448.89,2048.0,128.00,Large
5,Llama 3.1 405B,405,NaN,Dense,H100 80GB HBM3,579.14,5000.0,500.00,Large
6,Llama 3.1 405B,405,NaN,Dense,H100 80GB HBM3,3661.85,500.0,2000.00,Large
7,Llama 3.1 405B,405,NaN,Dense,H100 80GB HBM3,4572.23,128.0,2048.00,Large
8,Llama 3.1 70B,70,NaN,Dense,H100 80GB HBM3,14686.01,128.0,128.00,Medium
9,Llama 3.1 70B,70,NaN,Dense,H100 80GB HBM3,17463.99,128.0,2048.00,Medium


In [20]:
# ============================================================
# EXTERNAL REFERENCE MODELS FOR CLASSES MISSING FROM JOULE
# ============================================================
#
# The Joule CSV mainly contains Medium and Large models.
#
# Therefore, for Nano / Micro / Small models we supplement
# the Joule database with external H100 throughput benchmarks.
#
# IMPORTANT:
# These values are REFERENCE THROUGHPUTS.
# They are NOT the actual TPS of the black-box model.
#
# The assumption is:
#
#   black-box model
#       -> HHAI size class
#       -> representative model in same class
#       -> use representative model's TPS
#
# This is intentionally approximate because this is a
# college-project black-box estimator.
# ============================================================


EXTERNAL_REFERENCES = {

    # --------------------------------------------------------
    # NANO: < 7B
    # --------------------------------------------------------
    #
    # Reference:
    # Llama 3.2 3B running on one H100.
    #
    # Recent H100 benchmark measurements show very high
    # aggregate throughput for this small model.
    #
    # We use a representative decoding throughput rather
    # than claiming this is universal for all <7B models.
    #
    "Nano": {
        "reference_model": "Llama 3.2 3B",
        "model_type": "Dense",
        "hardware": "1x H100",
        "gpu_count": 1,

        # Representative H100 aggregate decode throughput.
        # Approximation selected from published H100 benchmark.
        "tps": 18_000,

        "source": "External H100 benchmark",

        "assumption": (
            "Llama 3.2 3B used as representative "
            "for <7B black-box models."
        )
    },


    # --------------------------------------------------------
    # MICRO: 7B - 20B
    # --------------------------------------------------------
    #
    # Reference:
    # Llama 3.1 8B
    #
    # NVIDIA reports:
    #
    # 1000 input / 1000 output
    # 1x H100
    # 15,270 output tokens/sec
    #
    # This is a particularly useful reference because the
    # hardware boundary is clearly one H100.
    #
    "Micro": {
        "reference_model": "Llama 3.1 8B",
        "model_type": "Dense",
        "hardware": "1x H100",
        "gpu_count": 1,

        "tps": 15_270,

        "source": "NVIDIA H100 inference benchmark",

        "assumption": (
            "Llama 3.1 8B used as representative "
            "for 7B-20B black-box models."
        )
    },


    # --------------------------------------------------------
    # SMALL: 20B - 40B
    # --------------------------------------------------------
    #
    # There is less clean matched dense-model H100 benchmark
    # coverage in this category.
    #
    # We therefore use a conservative approximate TPS between
    # the Micro and Medium reference classes.
    #
    # THIS IS OUR PROJECT ASSUMPTION, NOT A PAPER VALUE.
    #
    "Small": {
        "reference_model": "20B-40B reference model",
        "model_type": "Approximate",
        "hardware": "2x H100",
        "gpu_count": 2,

        "tps": 10_000,

        "source": "Project approximation",

        "assumption": (
            "Approximate throughput used because a clean "
            "matched dense 20B-40B H100 benchmark was not "
            "available in our selected benchmark sources."
        )
    }
}

In [11]:
def get_reference_tps(
    model_class,
    input_tokens,
    output_tokens,
    top_k=5
):
    """
    Estimate aggregate production TPS for a black-box model.

    METHOD:
    -------
    1. Filter Joule database to models in the same HHAI class.
    2. Prefer H100 benchmark records.
    3. Find records with input/output token lengths closest
       to the current query.
    4. Use the median TPS of the nearest records.


    ASSUMPTION:
    -----------
    TPS from open models in the same broad model class can be
    used as a rough proxy for an unknown black-box model.

    This is NOT the black-box model's measured TPS.
    """

    subset = joule_df[
        joule_df["HHAI_Class"] == model_class
    ].copy()


    # Joule's released dataset primarily targets H100 systems.
    h100_subset = subset[
        subset["Hardware"]
        .astype(str)
        .str.contains(
            "H100",
            case=False,
            na=False
        )
    ]

    if len(h100_subset) > 0:
        subset = h100_subset


    if len(subset) == 0:
        raise ValueError(
            f"No Joule TPS records found "
            f"for class {model_class}"
        )


    # Make sure columns are numeric.
    for column in [
        "TPS",
        "Input Length",
        "Output Length"
    ]:

        subset[column] = pd.to_numeric(
            subset[column],
            errors="coerce"
        )


    subset = subset.dropna(
        subset=[
            "TPS",
            "Input Length",
            "Output Length"
        ]
    )


    # --------------------------------------------------------
    # QUERY-SIZE DISTANCE
    # --------------------------------------------------------
    #
    # Use log distance because token counts can range from
    # hundreds to thousands.
    # --------------------------------------------------------

    subset["distance"] = (

        np.abs(
            np.log1p(subset["Input Length"])
            -
            np.log1p(input_tokens)
        )

        +

        np.abs(
            np.log1p(subset["Output Length"])
            -
            np.log1p(output_tokens)
        )
    )


    nearest = (
        subset
        .sort_values("distance")
        .head(top_k)
    )


    # Median is used instead of mean to reduce the effect
    # of one unusually high/low benchmark.
    reference_tps = nearest["TPS"].median()


    return float(reference_tps), nearest

In [23]:
def get_blackbox_reference(
    model_class,
    input_tokens,
    output_tokens
):
    """
    Select throughput reference for a black-box model.

    Nano / Micro / Small:
        Use our external reference table.

    Medium / Large:
        Use the Joule TPS database.

    IMPORTANT:
    All TPS values are proxies for the unknown black-box
    model's production throughput.
    """

    # ========================================================
    # NANO / MICRO / SMALL
    # ========================================================

    if model_class in EXTERNAL_REFERENCES:

        ref = EXTERNAL_REFERENCES[model_class]

        return {
            "reference_model":
                ref["reference_model"],

            "hardware":
                ref["hardware"],

            "gpu_count":
                ref["gpu_count"],

            "tps":
                ref["tps"],

            "source":
                ref["source"],

            "assumption":
                ref["assumption"],

            "reference_rows":
                None
        }


    # ========================================================
    # MEDIUM / LARGE
    # ========================================================
    #
    # Use Joule CSV because these classes have sufficient
    # benchmark records.
    # ========================================================

    reference_tps, rows = get_reference_tps(
        model_class=model_class,
        input_tokens=input_tokens,
        output_tokens=output_tokens
    )


    return {
        "reference_model":
            "Joule same-class models",

        "hardware":
            "H100 reference",

        "gpu_count":
            CLASS_GPU_COUNT[model_class],

        "tps":
            reference_tps,

        "source":
            "Oviedo et al. Joule TPS database",

        "assumption": (
            "Median TPS of nearest benchmark workloads "
            "from models in the same HHAI size class."
        ),

        "reference_rows":
            rows
    }

In [12]:
# ============================================================
# H100 HARDWARE POWER MODEL
# ============================================================
#
# Joule:
#
#     8 × H100 node
#     maximum node power ≈ 10.2 kW
#
# HHAI classes assume:
#
#     Nano   -> 1 GPU
#     Micro  -> 1 GPU
#     Small  -> 2 GPUs
#     Medium -> 4 GPUs
#     Large  -> 8 GPUs
#
#
# OUR PROJECT ASSUMPTION:
#
# For smaller classes, maximum node power is approximated
# by scaling the 8-GPU Joule node power proportionally.
#
# This is a simplification because CPU, memory, networking,
# cooling etc. do NOT scale perfectly with GPU count.
# ============================================================


H100_8GPU_MAX_POWER_KW = 10.2


def estimate_max_power(gpu_count):

    max_power_kw = (
        H100_8GPU_MAX_POWER_KW
        * gpu_count
        / 8
    )

    return max_power_kw

In [13]:
# ============================================================
# MONTE CARLO ASSUMPTIONS
# ============================================================
#
# Borrowed from Joule:
#
# Power P5  ≈ 40% of Pmax
# Power P95 ≈ 90% of Pmax
#
# PUE P5  = 1.05
# PUE P95 = 1.40
#
# 10,000 Monte Carlo simulations.
# ============================================================


POWER_P5_FRACTION = 0.40
POWER_P95_FRACTION = 0.90

PUE_P5 = 1.05
PUE_P95 = 1.40


def sample_lognormal_from_p5_p95(
    p5,
    p95,
    n_samples,
    rng
):

    z = 1.6448536269514722

    mu = (
        np.log(p5)
        +
        np.log(p95)
    ) / 2

    sigma = (
        np.log(p95)
        -
        np.log(p5)
    ) / (2 * z)

    return rng.lognormal(
        mean=mu,
        sigma=sigma,
        size=n_samples
    )

In [14]:
def estimate_energy(
    output_tokens,
    production_tps,
    max_power_kw,
    n_samples=N_SAMPLES,
    seed=RANDOM_SEED
):
    """
    Joule-style energy estimate.

    E = PUE × Power × EffectiveTokens / TPS


    ASSUMPTIONS:
    ------------

    1. Effective tokens ≈ output tokens.

       This follows Joule's decoding-dominated approximation.

    2. TPS is NOT measured for the black-box model.

       It is borrowed from comparable H100 models in the
       same HHAI category.

    3. Power utilization is modeled using the Joule-inspired
       P5/P95 uncertainty range.

    4. PUE is unknown and sampled from the Joule range.

    5. Efficiency multiplier alpha = 1.
    """

    rng = np.random.default_rng(seed)


    # Power uncertainty
    power_samples = (
        sample_lognormal_from_p5_p95(
            p5=max_power_kw
               * POWER_P5_FRACTION,

            p95=max_power_kw
                * POWER_P95_FRACTION,

            n_samples=n_samples,
            rng=rng
        )
    )


    # PUE uncertainty
    pue_samples = (
        sample_lognormal_from_p5_p95(
            p5=PUE_P5,
            p95=PUE_P95,
            n_samples=n_samples,
            rng=rng
        )
    )


    # --------------------------------------------------------
    # ENERGY CALCULATION
    # --------------------------------------------------------
    #
    # output_tokens / TPS = attributable serving time (seconds)
    #
    # power kW × seconds
    # converted to Wh using ×1000/3600.
    # --------------------------------------------------------

    attributable_seconds = (
        output_tokens
        /
        production_tps
    )


    energy_wh = (
        power_samples
        *
        attributable_seconds
        *
        pue_samples
        *
        1000
        /
        3600
    )


    return energy_wh

In [15]:
# ============================================================
# CARBON ASSUMPTION
# ============================================================
#
# Carbon formula:
#
#     Carbon (kg CO2e)
#          = Energy (kWh) × CIF
#
# Equivalent when energy is Wh and output is grams:
#
#     Carbon (g CO2e)
#          = Energy (Wh) × CIF
#
#
# ASSUMPTION:
#
# We do not know which physical data center processed a
# black-box query.
#
# Therefore this CIF is a reference value, NOT the actual
# real-time grid carbon intensity.
# ============================================================


CARBON_INTENSITY_KG_PER_KWH = 0.3528


def energy_to_carbon(
    energy_wh,
    carbon_intensity=CARBON_INTENSITY_KG_PER_KWH
):

    return (
        energy_wh
        *
        carbon_intensity
    )

In [16]:
def summarize_distribution(samples):

    return {
        "P5": np.percentile(samples, 5),
        "P25": np.percentile(samples, 25),
        "P50": np.percentile(samples, 50),
        "P75": np.percentile(samples, 75),
        "P95": np.percentile(samples, 95)
    }

In [21]:
def estimate_blackbox_model(
    model_tag,
    input_tokens,
    output_tokens,
    carbon_intensity=CARBON_INTENSITY_KG_PER_KWH
):

    # ========================================================
    # STEP 1
    # HHAI-STYLE MODEL CLASSIFICATION
    # ========================================================
    #
    # We classify the black-box model as:
    #
    # Nano / Micro / Small / Medium / Large
    #
    # If parameter count is visible in the model tag,
    # we use it.
    #
    # Otherwise we use the HHAI-inspired naming heuristic.
    # ========================================================

    model_info = classify_blackbox_model(
        model_tag
    )

    model_class = model_info["class"]
    gpu_count = model_info["gpu_count"]


    # ========================================================
    # STEP 2
    # ESTIMATE HARDWARE POWER
    # ========================================================
    #
    # We assume H100 hardware.
    #
    # GPU count comes from the HHAI class:
    #
    # Nano   -> 1 GPU
    # Micro  -> 1 GPU
    # Small  -> 2 GPUs
    # Medium -> 4 GPUs
    # Large  -> 8 GPUs
    #
    # Power is estimated by scaling the Joule
    # 8-H100 node power.
    # ========================================================

    max_power_kw = estimate_max_power(
        gpu_count
    )


    # ========================================================
    # STEP 3
    # SELECT A REFERENCE TPS
    # ========================================================
    #
    # GPT/Claude/etc. production TPS is unknown.
    #
    # Therefore:
    #
    # Nano / Micro / Small
    #     -> use EXTERNAL_REFERENCES
    #
    # Medium / Large
    #     -> use comparable models from Joule CSV
    #
    # IMPORTANT:
    # This TPS is a PROXY.
    # It is NOT measured TPS of the black-box model.
    # ========================================================

    reference = get_blackbox_reference(
        model_class=model_class,
        input_tokens=input_tokens,
        output_tokens=output_tokens
    )

    reference_tps = reference["tps"]


    # ========================================================
    # STEP 4
    # MONTE CARLO ENERGY ESTIMATION
    # ========================================================
    #
    # Uses Joule-style:
    #
    # Energy =
    #     Power
    #     × Output Tokens / TPS
    #     × PUE
    #
    # Power and PUE are uncertain, so we generate
    # 10,000 possible scenarios.
    # ========================================================

    energy_samples = estimate_energy(
        output_tokens=output_tokens,
        production_tps=reference_tps,
        max_power_kw=max_power_kw
    )


    energy_summary = summarize_distribution(
        energy_samples
    )


    # ========================================================
    # STEP 5
    # CARBON ESTIMATION
    # ========================================================
    #
    # Carbon:
    #
    #     gCO2e = Energy(Wh) × CIF(kgCO2e/kWh)
    #
    # CIF is an assumed/reference grid carbon intensity,
    # NOT necessarily the actual grid used by the provider.
    # ========================================================

    carbon_samples = energy_to_carbon(
        energy_samples,
        carbon_intensity
    )


    carbon_summary = summarize_distribution(
        carbon_samples
    )


    # ========================================================
    # RETURN EVERYTHING
    # ========================================================

    return {

        # Input model
        "model": model_tag,

        # HHAI classification
        "class": model_class,

        "classification_method":
            model_info["classification_method"],

        # Hardware assumptions
        "gpu_count": gpu_count,

        "max_power_kw": max_power_kw,

        # TPS reference information
        "reference_model":
            reference["reference_model"],

        "reference_tps":
            reference_tps,

        "reference_source":
            reference["source"],

        "reference_assumption":
            reference["assumption"],

        # Results
        "energy_wh":
            energy_summary,

        "carbon_gco2e":
            carbon_summary,

        # If Medium/Large, this contains the
        # actual Joule rows used.
        #
        # For Nano/Micro/Small this will be None.
        "reference_rows":
            reference["reference_rows"]
    }

In [26]:
result = estimate_blackbox_model(
    model_tag="llama-3.1-8b",
    input_tokens=500,
    output_tokens=300
)

In [27]:
print("BLACK-BOX MODEL ESTIMATE")
print("========================")

print("Model:", result["model"])
print("HHAI Class:", result["class"])
print("Classification:", result["classification_method"])

print("\nHARDWARE ASSUMPTION")
print("-------------------")
print("GPU Count:", result["gpu_count"])
print(f"Max Power: {result['max_power_kw']:.2f} kW")

print("\nTPS REFERENCE")
print("-------------")
print("Reference Model:", result["reference_model"])
print(f"Reference TPS: {result['reference_tps']:,.2f}")
print("Source:", result["reference_source"])
print("Assumption:", result["reference_assumption"])

print("\nENERGY")
print("------")

for p, value in result["energy_wh"].items():
    print(f"{p}: {value:.4f} Wh")

print("\nCARBON")
print("------")

for p, value in result["carbon_gco2e"].items():
    print(f"{p}: {value:.4f} gCO2e")

BLACK-BOX MODEL ESTIMATE
Model: llama-3.1-8b
HHAI Class: Micro
Classification: Parameter count found in model tag (8.0B)

HARDWARE ASSUMPTION
-------------------
GPU Count: 1
Max Power: 1.27 kW

TPS REFERENCE
-------------
Reference Model: Llama 3.1 8B
Reference TPS: 15,270.00
Source: NVIDIA H100 inference benchmark
Assumption: Llama 3.1 8B used as representative for 7B-20B black-box models.

ENERGY
------
P5: 0.0033 Wh
P25: 0.0043 Wh
P50: 0.0051 Wh
P75: 0.0060 Wh
P95: 0.0078 Wh

CARBON
------
P5: 0.0012 gCO2e
P25: 0.0015 gCO2e
P50: 0.0018 gCO2e
P75: 0.0021 gCO2e
P95: 0.0028 gCO2e
